In [20]:
# Step 1: Parse movie_lines.txt
id2line = {}
with open("movie_lines.txt", encoding="ISO-8859-1") as f:  # You can try changing this and see what changes in the response you get :)
    for line in f:
        parts = line.strip().split(" +++$+++ ")
        if len(parts) == 5:
            line_id = parts[0]
            text = parts[4]
            id2line[line_id] = text

# Step 2: Parse movie_conversations.txt into a list of conversations
conversations = []
with open("movie_conversations.txt", encoding="ISO-8859-1") as f:  # Make sure to keet encodings consistent
    for line in f:
        parts = line.strip().split(" +++$+++ ")
        if len(parts) == 4:
            try:
                utterance_ids = eval(parts[3])  # Converts string list to actual list
                conversations.append(utterance_ids)
            except Exception as e:
                print(f"Skipping line due to eval error: {e}")

# Step 3: Build input-output pairs (prompt-response)
pairs = []
for conv in conversations:
    for i in range(len(conv) - 1):
        if conv[i] in id2line and conv[i+1] in id2line:
            input_line = id2line[conv[i]].strip()
            target_line = id2line[conv[i+1]].strip()
            if input_line and target_line:  # skip empty lines
                pairs.append((input_line, target_line))


print(f"Loaded {len(pairs)} dialog pairs.")


Loaded 221282 dialog pairs.


In [21]:
import random

# Parameters
SAMPLE_SIZE = 5_000      # how many pairs you want, you can change it
RANDOM_SEED = 42          # set this if you need deterministic sampling

# Draw the sample
random.seed(RANDOM_SEED)          # comment this out for a fresh shuffle each run
sample_pairs = random.sample(pairs, SAMPLE_SIZE)

print(f"Sampled {len(sample_pairs)} pairs.")

Sampled 5000 pairs.


In [22]:
from datasets import Dataset

# Create a Hugging Face Dataset from your list of (input, output) pairs
data = [{"input": q, "output": a} for q, a in sample_pairs]
hf_dataset = Dataset.from_list(data)

print(hf_dataset[0])  # sanity check


{'input': "You didn't come here to destroy Wintermute. You can to save a man you love. A man who isn't even capable of returning that love. Such a waste...", 'output': "My man's coming to get my ass out of here. That's good enough for me."}


In [23]:
# If not installed, uncomment the code and install
# !pip install transformers


In [24]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")
tokenizer.pad_token = tokenizer.eos_token  # Fix the pad token issue


def tokenize(example):
    input_text = example["input"] + tokenizer.eos_token
    output_text = example["output"] + tokenizer.eos_token
    full_text = input_text + output_text
    tokens = tokenizer(full_text, truncation=True, padding="max_length", max_length=128)
    tokens["labels"] = tokens["input_ids"].copy()  # Causal language modeling
    return tokens

tokenized_dataset = hf_dataset.map(tokenize, batched=False)


Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [25]:
# If error persists, uncomment this code, execute it and run below snippet again
# !pip install -U transformers


In [26]:
import os
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
)
# Need to import get_last_checkpoint from the trainer_utils module
from transformers.trainer_utils import get_last_checkpoint


# 1. Detect an existing checkpoint (if any)

output_dir = "./dialogpt-finetuned"
last_ckpt  = get_last_checkpoint(output_dir) if os.path.isdir(output_dir) else None
if last_ckpt:
    print(f"  Found checkpoint at: {last_ckpt} – resuming from there.")


# 2. Load model (fresh or from checkpoint)

model_name_or_path = last_ckpt or "microsoft/DialoGPT-medium"
model = AutoModelForCausalLM.from_pretrained(model_name_or_path)

# (Optional but tidy) – make sure pad token is set

tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
tokenizer.pad_token = tokenizer.eos_token


training_args = TrainingArguments(
    output_dir           = output_dir,
    per_device_train_batch_size = 4,
    num_train_epochs     = 2,
    dataloader_num_workers = 4,

    # logging & checkpointing
    logging_strategy     = "steps",
    logging_steps        = 200,
    save_strategy        = "steps",
    save_steps           = 500,
    save_total_limit     = 2,

    # misc
    fp16                 = True,     # comment out if GPU doesn’t support fp16
    report_to            = "none",   # no WandB/HF Hub logging
)


# 4. Trainer
trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized_dataset,
    tokenizer     = tokenizer,  # keeps pad/eos alignment neat
)


# 5. Train – resume if we have a checkpoint
trainer.train(resume_from_checkpoint=last_ckpt)

  Found checkpoint at: ./dialogpt-finetuned/checkpoint-2500 – resuming from there.


/tmp/ipython-input-4023397890.py:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Step,Training Loss


TrainOutput(global_step=2500, training_loss=0.0, metrics={'train_runtime': 0.0019, 'train_samples_per_second': 5294501.389, 'train_steps_per_second': 1323625.347, 'total_flos': 2321751736320000.0, 'train_loss': 0.0, 'epoch': 2.0})

In [27]:
# Saving the freshly trained moel and its tokeniser
trainer.save_model("./dialogpt-finetuned/final")
tokenizer.save_pretrained("./dialogpt-finetuned/final")


('./dialogpt-finetuned/final/tokenizer_config.json',
 './dialogpt-finetuned/final/special_tokens_map.json',
 './dialogpt-finetuned/final/chat_template.jinja',
 './dialogpt-finetuned/final/vocab.json',
 './dialogpt-finetuned/final/merges.txt',
 './dialogpt-finetuned/final/added_tokens.json',
 './dialogpt-finetuned/final/tokenizer.json')

In [28]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the fine-tuned model
model_path = "./dialogpt-finetuned/final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model     = AutoModelForCausalLM.from_pretrained(model_path)
model.eval()

# FIX: Initialize history as a clean tensor
chat_history_ids = torch.tensor([], dtype=torch.long)

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]: break

    # 1. Encode user input WITHOUT adding a trailing EOS here (the model adds its own)
    new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

    # 2. Append to history and TRUNCATE (The Secret Sauce)
    # If history is too long, DialoGPT just returns empty. We keep only the last 128 tokens.
    if chat_history_ids.shape[-1] > 0:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    if bot_input_ids.shape[-1] > 256:
        bot_input_ids = bot_input_ids[:, -256:]

    # 3. Create an Attention Mask (Crucial for multi-turn)
    attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

    # 4. GENERATE - Forcing the model to speak
    output_ids = model.generate(
        bot_input_ids,
        attention_mask=attention_mask,
        max_new_tokens=50,
        min_new_tokens=2,               # <--- FORCE at least 2 words
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
        do_sample=True,
        top_k=50,
        top_p=0.9,
        temperature=0.8,
        repetition_penalty=1.5          # <--- High penalty to stop "silent" loops
    )

    # 5. Slicing logic
    response_ids = output_ids[:, bot_input_ids.shape[-1]:]
    response = tokenizer.decode(response_ids[0], skip_special_tokens=True)

    # 6. Final UI output
    if not response.strip():
        # Last resort: Try generating without history to see if model is alive
        print("Bot: (Thinking...)")
    else:
        print(f"Bot: {response}")

    # 7. Update history
    chat_history_ids = output_ids

KeyboardInterrupt: Interrupted by user

# **This is the snippet to use Dialogpt-medium, for those who are stuck try running this cell first to get a clearer idea about how to proceed and code**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model     = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium").eval()

chat_history = []

while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]: break

    new_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors="pt")
    bot_ids = torch.cat(chat_history + [new_ids], dim=-1) if chat_history else new_ids

    generated_ids = model.generate(
        bot_ids,
        max_length=bot_ids.shape[-1] + 100,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,
        temperature=0.7,
        top_k=40,
        top_p=0.9,
    )

    reply = tokenizer.decode(generated_ids[:, bot_ids.shape[-1]:][0],
                             skip_special_tokens=True)
    print(f"Bot: {reply}")
    chat_history.append(new_ids)


You: You killed my brother! You cowardly son of a gun! Gunned him down when he wasn't hardly looking.
Bot: He was a coward! He didn't even know the difference between the two words!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')